This notebook parses and imputes the annotations in the file created in the Watershed-SNV workflow.

This notebook should take about 45 seconds to run.

## Setup

In [1]:
library(data.table)
library(AnVILGCP)
options(datatable.na.strings=c('NA','.'))

watershed_snv_run_tbl <- setDT(avtable('example_watershed_snv_run'))

# Copies the annotated BCF output of the workflow to the virtual environment.
# If you ran the workflow on multiple of the subsets of chromosomes, the larger subset will be prioritized.
system(paste(
  'gcloud storage cp',
  watershed_snv_run_tbl[example_watershed_snv_run_id=='MAGE_1000G',                  annotated_bcf],
  watershed_snv_run_tbl[example_watershed_snv_run_id=='MAGE_1000G_chr18_chr19_chr20',annotated_bcf],
  watershed_snv_run_tbl[example_watershed_snv_run_id=='MAGE_1000G_chr22_chr21',      annotated_bcf],
  '.'
))

Registered S3 methods overwritten by 'AnVIL':
  method                         from    
  print.avworkflow_configuration AnVILGCP
  print.gcloud_sdk_result        AnVILGCP



We only care about genes with expression data. We make a file of these genes' names which we will use to filter by.

In [2]:
z <- fread('indiv_gene_z.tsv', stringsAsFactors=T)
writeLines(sub('\\..*','',unique(z$gene)),'expression_genes.txt')

Define desired fields to pull from the annotated BCF file output by the workflow:

In [3]:
fields <- c(
  'CHROM','POS','REF','ALT', 'MAF',
  'GC','CpG',
  'SIFTcat','SIFTval',
  'PolyPhenCat','PolyPhenVal',
  'priPhCons','mamPhCons','verPhCons',
  'priPhyloP','mamPhyloP','verPhyloP',
  'bStatistic','GerpN','GerpS','PHRED',
  'LoF','phyloP100way','Consequence'
)

f_flag <- paste(collapse='\t', paste0('%',fields)) # '%CHROM\t%POS\t%REF ...' etc.

In [4]:
d <- fread(cmd=paste0( # Pull from the annotated BCF file output by the WDL
  '< annotated.bcf.gz',
  ' bcftools view -i"AC<0.5*AN" -Ou',
  ' | bcftools norm -d snps -Ou', # rm duplicate variants
  ' | bcftools +split-vep',
  '   -g expression_genes.txt -f "',f_flag,'"'
), col.names=fields, stringsAsFactors=T)

Why filter by `AC<0.5*AN`?
We only care about rare variants. Only the ALT allele is annotated by VEP.
But sometimes ALT is the major (not-rare) allele -- we don't care about these annotations.

In [5]:
head(d,3)

CHROM,POS,REF,ALT,MAF,GC,CpG,SIFTcat,SIFTval,PolyPhenCat,⋯,priPhyloP,mamPhyloP,verPhyloP,bStatistic,GerpN,GerpS,PHRED,LoF,phyloP100way,Consequence
<fct>,<int>,<fct>,<fct>,<dbl>,<dbl>,<dbl>,<fct>,<dbl>,<fct>,⋯,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>
chr15,20123785,C,T,0.00462963,0.563,0.013,NA,NA,NA,⋯,0.365,0.363,0.351,1000,0.233,-0.465,3.575,NA,-0.587000012397766,downstream_gene_variant
chr15,20123786,G,T,0.00185185,0.556,0.013,NA,NA,NA,⋯,0.365,0.363,0.351,1000,0.233,0.233,3.445,NA,0.331999987363815,downstream_gene_variant
chr15,20123795,T,C,0.00462963,0.550,0.013,NA,NA,NA,⋯,-1.943,-1.928,-1.912,1000,0.233,-0.465,1.755,NA,-1.89499998092651,downstream_gene_variant


## Calculate nearest TSS / TES distance using GENCODE

In [6]:
FLANK <- 1e4

In [7]:
g <- fread( # Read the GENCODE file, but only the rows whose feature type is 'gene'
  cmd='zgrep -P "\tgene\t" gencode.v49.annotation.gtf.gz',
  col.names=c('CHROM','source','feature','start','end','score','strand','frame','attribute')
)

In [8]:
head(g,3)

CHROM,source,feature,start,end,score,strand,frame,attribute
<chr>,<chr>,<chr>,<int>,<int>,<lgl>,<chr>,<lgl>,<chr>
chr1,HAVANA,gene,11121,24894,NA,+,NA,"gene_id ""ENSG00000290825.2""; gene_type ""lncRNA""; gene_name ""DDX11L16""; level 2; tag ""overlaps_pseudogene"";"
chr1,HAVANA,gene,12010,13670,NA,+,NA,"gene_id ""ENSG00000223972.6""; gene_type ""transcribed_unprocessed_pseudogene""; gene_name ""DDX11L1""; level 2; hgnc_id ""HGNC:37102""; havana_gene ""OTTHUMG00000000961.2"";"
chr1,HAVANA,gene,14356,30744,NA,-,NA,"gene_id ""ENSG00000310526.1""; gene_type ""lncRNA""; gene_name ""WASH7P""; level 2; hgnc_id ""HGNC:38034""; tag ""ncRNA_host""; tag ""overlapping_locus""; tag ""overlaps_pseudogene"";"


Extract the `gene_id` from the `attribute` column of the GENCODE file, filter it to our list of genes which have expression data, join each gene to the annotated SNVs falling within the flank of that gene, and finally for each SNV take the TSS/TES which is the minimum distance away.  

In [9]:
g <- g[
  ][, gene_id := sub('.*"(.*)"', '\\1', tstrsplit(attribute,';')[[1]]) # 'gene_id "ENSG###.#" ...' --> 'ENSG###.#'
  ][, gene_id := sub(   '\\..*',    '', gene_id                      ) # 'ENSG###.#' --> 'ENSG###'
  ][gene_id %in% readLines('expression_genes.txt')
  ][, `:=`(startflank = pmax(start - FLANK, 1),
             endflank =        end + FLANK,
              tss_pos = fifelse(strand=='+', start, end),
              tes_pos = fifelse(strand=='+', end, start))

  # Join genes to SNVs within that gene's startflank & endflank, then calc nearest TSS/TES distance per SNV.
  ][d, on=.(CHROM, startflank<POS, endflank>POS), nomatch=NULL
    ,    .(CHROM,POS,REF,ALT, tss_pos, tes_pos)
  ][, by=.(CHROM,POS,REF,ALT)
    , .(nearest_tss_dist = min(abs(tss_pos-POS)),
        nearest_tes_dist = min(abs(tes_pos-POS)))
]

In [10]:
# Add this new "nearest TSS/TES" annotation by joining to the original df.
d <- d[g, on=.(CHROM,POS,REF,ALT)]

## Convert categorical inputs to indicator variables

In [11]:
d <- d[
  ][, phyloP100way                        := tstrsplit(phyloP100way,',')[[1]] |> as.numeric() # NA warnings are due to '.' entries.

  ][, LoF_HC                              := grepl('HC'                                 , LoF        , fixed=T)
  ][, LoF_LC                              := grepl('LC'                                 , LoF        , fixed=T)

  ][, SIFTcat_tolerated                   := grepl('tolerated'                          , SIFTcat    , fixed=T)
  ][, SIFTcat_deleterious                 := grepl('deleterious'                        , SIFTcat    , fixed=T)

  ][, PolyPhen_benign                     := grepl('benign'                             , PolyPhenCat, fixed=T)
  ][, PolyPhen_possibly_damaging          := grepl('possibly'                           , PolyPhenCat, fixed=T)
  ][, PolyPhen_probably_damaging          := grepl('probably'                           , PolyPhenCat, fixed=T)
  ][, PolyPhen_unknown                    := grepl('unknown'                            , PolyPhenCat, fixed=T)

  ][, transcript_ablation                 := grepl('transcript_ablation'                , Consequence, fixed=T)
  ][, splice_acceptor_variant             := grepl('splice_acceptor_variant'            , Consequence, fixed=T)
  ][, splice_donor_variant                := grepl('splice_donor_variant'               , Consequence, fixed=T)
  ][, stop_gained                         := grepl('stop_gained'                        , Consequence, fixed=T)
  ][, frameshift_variant                  := grepl('frameshift_variant'                 , Consequence, fixed=T)
  ][, stop_lost                           := grepl('stop_lost'                          , Consequence, fixed=T)
  ][, start_lost                          := grepl('start_lost'                         , Consequence, fixed=T)
  ][, transcript_amplification            := grepl('transcript_amplification'           , Consequence, fixed=T)
  ][, feature_elongation                  := grepl('feature_elongation'                 , Consequence, fixed=T)
  ][, feature_truncation                  := grepl('feature_truncation'                 , Consequence, fixed=T)
  ][, inframe_insertion                   := grepl('inframe_insertion'                  , Consequence, fixed=T)
  ][, inframe_deletion                    := grepl('inframe_deletion'                   , Consequence, fixed=T)
  ][, missense_variant                    := grepl('missense_variant'                   , Consequence, fixed=T)
  ][, protein_altering_variant            := grepl('protein_altering_variant'           , Consequence, fixed=T)
  ][, splice_donor_5th_base_variant       := grepl('splice_donor_5th_base_variant'      , Consequence, fixed=T)
  ][, splice_region_variant               := grepl('splice_region_variant'              , Consequence, fixed=T)
  ][, splice_donor_region_variant         := grepl('splice_donor_region_variant'        , Consequence, fixed=T)
  ][, splice_polypyrimidine_tract_variant := grepl('splice_polypyrimidine_tract_variant', Consequence, fixed=T)
  ][, incomplete_terminal_codon_variant   := grepl('incomplete_terminal_codon_variant'  , Consequence, fixed=T)
  ][, start_retained_variant              := grepl('start_retained_variant'             , Consequence, fixed=T)
  ][, stop_retained_variant               := grepl('stop_retained_variant'              , Consequence, fixed=T)
  ][, synonymous_variant                  := grepl('synonymous_variant'                 , Consequence, fixed=T)
  ][, coding_sequence_variant             := grepl('coding_sequence_variant'            , Consequence, fixed=T)
  ][, mature_miRNA_variant                := grepl('mature_miRNA_variant'               , Consequence, fixed=T)
  ][, `5_prime_UTR_variant`               := grepl('5_prime_UTR_variant'                , Consequence, fixed=T)
  ][, `3_prime_UTR_variant`               := grepl('3_prime_UTR_variant'                , Consequence, fixed=T)
  ][, non_coding_transcript_exon_variant  := grepl('non_coding_transcript_exon_variant' , Consequence, fixed=T)
  ][, intron_variant                      := grepl('intron_variant'                     , Consequence, fixed=T)
  ][, NMD_transcript_variant              := grepl('NMD_transcript_variant'             , Consequence, fixed=T)
  ][, non_coding_transcript_variant       := grepl('non_coding_transcript_variant'      , Consequence, fixed=T)
  ][, coding_transcript_variant           := grepl('[^_]coding_transcript_variant'      , Consequence         )
  ][, upstream_gene_variant               := grepl('upstream_gene_variant'              , Consequence, fixed=T)
  ][, downstream_gene_variant             := grepl('downstream_gene_variant'            , Consequence, fixed=T)
  ][, TFBS_ablation                       := grepl('TFBS_ablation'                      , Consequence, fixed=T)
  ][, TFBS_amplification                  := grepl('TFBS_amplification'                 , Consequence, fixed=T)
  ][, TF_binding_site_variant             := grepl('TF_binding_site_variant'            , Consequence, fixed=T)
  ][, regulatory_region_ablation          := grepl('regulatory_region_ablation'         , Consequence, fixed=T)
  ][, regulatory_region_amplification     := grepl('regulatory_region_amplification'    , Consequence, fixed=T)
  ][, regulatory_region_variant           := grepl('regulatory_region_variant'          , Consequence, fixed=T)
  ][, intergenic_variant                  := grepl('intergenic_variant'                 , Consequence, fixed=T)
  ][, sequence_variant                    := grepl('sequence_variant'                   , Consequence, fixed=T)

  ][, names(.SD) := lapply(.SD, as.integer), .SDcols=is.logical
]

Warning message in eval(jsub, SDenv, parent.frame()):
“NAs introduced by coercion”


Don't mind the "NAs introduced by coersion" warning: it is due converting the phyloP100way scores to numeric despite there being some `.` entries (which represent NA anyway).

Below are the number of each type of SNV:

In [12]:
d[, data.table(type=names(.SD), count=lapply(.SD,sum)), .SDcols=is.integer]

type,count
<chr>,<list>
POS,1.194489e+14
bStatistic,NA
nearest_tss_dist,253236911460
nearest_tes_dist,257340072362
LoF_HC,333
LoF_LC,123
SIFTcat_tolerated,6158
SIFTcat_deleterious,4204
PolyPhen_benign,6599


## Join SNV annotations onto Gene-Individual pairs
First, we collect a list of all possible individual-gene-snv trios `a`; then we join that dataframe to the SNV annotations `d`. Then later we can easily collapse the annotations to the gene-individual pair level by grouping by gene & indivudal.

In [13]:
# Indiv-gene-snv trios
a <- fread(cmd=paste0(
  '< annotated.bcf.gz',
  ' bcftools view -i"AC<0.5*AN" -Ou',
  ' | bcftools norm -d snps -Ou',
  ' | bcftools +split-vep',
  '  -i\'GT=="alt"\'', # Filter to samples with at least 1 ALT allele
  '  -f"[%CHROM\t%POS\t%REF\t%ALT\t%Gene\t%SAMPLE\n]"',
  '  -d -g expression_genes.txt'
), col.names=c('CHROM','POS','REF','ALT','Gene','indiv'), stringsAsFactors=T)

In [14]:
a[order(Gene,indiv)] |> head()

CHROM,POS,REF,ALT,Gene,indiv
<fct>,<int>,<fct>,<fct>,<fct>,<fct>
chr7,121327973,T,C,ENSG00000002745,HG00105
chr7,121327973,T,C,ENSG00000002745,HG00105
chr7,121325117,A,C,ENSG00000002745,HG00108
chr7,121325117,A,C,ENSG00000002745,HG00108
chr7,121327973,T,C,ENSG00000002745,HG00114
chr7,121327973,T,C,ENSG00000002745,HG00114


Why are there duplicated rows? VEP adds an annotation set for every transcript the SNV affects. Many of said transcripts may affect the same gene. \
But we don't extract the transcript names, only the genes. Thus we see duplicate rows. \
In other words, if we had included an extra column showing transcript names, there would be no duplicate rows. \
We remove these duplicate rows.

In [15]:
a <- a[!duplicated(a)]
a[order(Gene,indiv)] |> head()

CHROM,POS,REF,ALT,Gene,indiv
<fct>,<int>,<fct>,<fct>,<fct>,<fct>
chr7,121327973,T,C,ENSG00000002745,HG00105
chr7,121325117,A,C,ENSG00000002745,HG00108
chr7,121327973,T,C,ENSG00000002745,HG00114
chr7,121343687,G,T,ENSG00000002745,HG00114
chr7,121320512,A,G,ENSG00000002745,HG00122
chr7,121320517,G,A,ENSG00000002745,HG00122


### Define N2 pairs
Watershed uses "N2 pairs" (individuals which share the same rare variant) to evaluate its performance. Each N2 pair is labeled with a unique integer. If there are >2 people sharing the same rare variant, the first two are arbitrarily chosen.

In [16]:
# Define N2 pairs
# (Per SNV, generate a unique id `.GRP`.
#  If >= 2 people have that SNV, assign the id to the first two.)
a <- a[, by=.(CHROM,POS,REF,ALT,Gene)
       , N2pair := fifelse(.N >= 2 & 1:.N <= 2, .GRP, NA)]

## Collapse annotations to gene-individual level
The collapsing is done with simple transformations defined in [Ferraro et al. Table S3](https://www.science.org/doi/suppl/10.1126/science.aaz5900/suppl_file/aaz5900-ferraro-sm-table-s3.xlsx). The gist is:

* If an individual has any e.g. missense rare variant near a gene, that gene-individual pair is labeled positive for missense. The same collapsing strategy is applied to all of the indicator column annotations describing variant types.
* The min TSS or TES distance is collapsed to the minimum TSS or TES distance of _any_ of the vairants near the gene.
* The count of rare variants an individual had near each gene is also calculated.

In [17]:
## Join the SNV-indiv-gene trios to the SNV annotations
m <- a[d, on=.(CHROM,POS,REF,ALT)]

## Most columns (like the indicator columns) we collapse to gene-individual level simply using max().
max_collapsed <- m[
  ][, by=.(Gene,indiv)
    , .SDcols=is.numeric
    , lapply(.SD,max,na.rm=T)
  ][, -c('POS','MAF','nearest_tss_dist','nearest_tes_dist')
]

## A few columns need different collapsing logic.
collapsed <- m[
  ][, by=.(Gene,indiv), .(
      minMAF = min(MAF),
      minTSSdist = min(nearest_tss_dist),
      minTESdist = min(nearest_tes_dist),
      n_rarevars_near_gene = nrow(.SD)  )

  # Merge with the max-collapsed things.
  ][on=.(Gene,indiv), max_collapsed

  # An indiv could have 2 diff SNVs in common w/ 2 diff ppl near the same gene.
  # When collapsing to gene-indiv level, you have to choose just 1 N2pair to keep.
  # Remove N2pairs that are left hanging (i.e. the N2pair id only has 1 occurence after collapsing).
  ][, by=N2pair, N2pair := fifelse(.N==1,NA,N2pair)
]

Now we join the collapsed annotation dataset to the expression z-scores and calculate p-values from them. \
Watershed allows us to make the p-value signed to preserve the information of whether the z-score was positive or negative (useful to know if a gene was under-expressed or ovre-expressed). 

In [18]:
# Merge in z-scores and calculate p-values
collapsed <- z[
 ][, .(indiv, Gene=sub('\\..*','',gene), z)
 ][collapsed, on=.(Gene,indiv)
 ][, p := sign(z)*2*pnorm(-abs(z))
]

## Annotation imputation
Any annotations which are missing are imputed according to [Ferraro et al. Table S3](https://www.science.org/doi/suppl/10.1126/science.aaz5900/suppl_file/aaz5900-ferraro-sm-table-s3.xlsx):

In [19]:
collapsed[, sapply(.SD, function(col) sum(is.na(col)))]
collapsed[
  ][is.na(        GC), GC         :=   0.418
  ][is.na(       CpG), CpG        :=   0.024
  ][is.na(bStatistic), bStatistic := 800
  ][is.na( priPhCons), priPhCons  :=   0.115
  ][is.na( mamPhCons), mamPhCons  :=   0.079
  ][is.na( verPhCons), verPhCons  :=   0.094
  ][is.na( priPhyloP), priPhyloP  :=  -0.033
  ][is.na( mamPhyloP), mamPhyloP  :=  -0.038
  ][is.na( verPhyloP), verPhyloP  :=   0.017
  ][is.na(     GerpN), GerpN      :=   1.909
  ][is.na(     GerpS), GerpS      :=  -0.2
]
collapsed[is.na(collapsed)] <- 0
all(collapsed[, sapply(.SD, function(col) sum(is.na(col)))]==0)
collapsed[N2pair==0, N2pair:=NA] # Except N2pair, those NAs should remain NA, not 0

indiv                                Gene 
                                  0                                   0 
                                  z                              minMAF 
                                  0                                   0 
                         minTSSdist                          minTESdist 
                                  0                                   0 
               n_rarevars_near_gene                              N2pair 
                                  0                              526151 
                                 GC                                 CpG 
                                  0                                   0 
                            SIFTval                         PolyPhenVal 
                             897347                              897060 
                          priPhCons                           mamPhCons 
                               5753                                5753 
                          verPhCons                           priPhyloP 
                               5753                                5753 
                          mamPhyloP                           verPhyloP 
                               5753                                5753 
                         bStatistic                               GerpN 
                              20837                                  11 
                              GerpS                               PHRED 
                                 11                               15782 
                       phyloP100way                              LoF_HC 
                               5753                                   0 
                             LoF_LC                   SIFTcat_tolerated 
                                  0                                   0 
                SIFTcat_deleterious                     PolyPhen_benign 
                                  0                                   0 
         PolyPhen_possibly_damaging          PolyPhen_probably_damaging 
                                  0                                   0 
                   PolyPhen_unknown                 transcript_ablation 
                                  0                                   0 
            splice_acceptor_variant                splice_donor_variant 
                                  0                                   0 
                        stop_gained                  frameshift_variant 
                                  0                                   0 
                          stop_lost                          start_lost 
                                  0                                   0 
           transcript_amplification                  feature_elongation 
                                  0                                   0 
                 feature_truncation                   inframe_insertion 
                                  0                                   0 
                   inframe_deletion                    missense_variant 
                                  0                                   0 
           protein_altering_variant       splice_donor_5th_base_variant 
                                  0                                   0 
              splice_region_variant         splice_donor_region_variant 
                                  0                                   0 
splice_polypyrimidine_tract_variant   incomplete_terminal_codon_variant 
                                  0                                   0 
             start_retained_variant               stop_retained_variant 
                                  0                                   0 
                 synonymous_variant             coding_sequence_variant 
                                  0                                   0 
               mature_miRNA_variant                 5_prime_UTR_variant 
               

[1] TRUE

## Formatting
The Watershed algorithm R scripts require the columns to have particular names and be in a particular order. See the [Watershed GitHub repository](https://github.com/BennyStrobes/Watershed?tab=readme-ov-file#input-file) for full details about the required input format.

In [20]:
# Watershed algorithm requires cols to be in a particular order:
# SubjectID, GeneName, <annotation(s)>, <p-value(s)>, N2pair
setcolorder(collapsed, c('indiv','Gene'))
setcolorder(collapsed, c('p','N2pair'), after=ncol(collapsed))
setnames(collapsed, c('indiv','Gene'), c('SubjectID','GeneName'))

Finally, we subset the columns to remove extra annotations that we collected that are superfluous to what was used in [Ferraro et al. Table S3](https://www.science.org/doi/suppl/10.1126/science.aaz5900/suppl_file/aaz5900-ferraro-sm-table-s3.xlsx).

In [21]:
tmp <- collapsed[, c(
  'SubjectID', 'GeneName',
  'minMAF','minTSSdist','minTESdist','n_rarevars_near_gene',
  'GC','CpG',
  'priPhCons','mamPhCons','verPhCons',
  'priPhyloP','mamPhCons','verPhCons',
  'bStatistic','GerpN','GerpS','PHRED',
  'LoF_HC','LoF_LC', 'phyloP100way',
  'SIFTval','SIFTcat_deleterious','SIFTcat_tolerated',
  'PolyPhenVal','PolyPhen_benign','PolyPhen_possibly_damaging','PolyPhen_probably_damaging','PolyPhen_unknown',
  '3_prime_UTR_variant','5_prime_UTR_variant',
  'downstream_gene_variant','upstream_gene_variant',
  'intron_variant',
  'missense_variant','synonymous_variant',
  'non_coding_transcript_exon_variant',
  'splice_acceptor_variant','splice_donor_variant','splice_region_variant',
  'stop_gained',
  #'TF_binding_site_variant',   # We have none
  #'intergenic_vairiant',       # We have none
  #'regulatory_region_variant', # We have none
  'p','N2pair'
)]

fwrite(collapsed, 'collapsed.tsv',         sep='\t',na='NA')
fwrite(tmp,       'collapsed-reduced.tsv', sep='\t',na='NA')